##Clone the Repo

In [1]:
# ============================================================
#  SETUP — Run this first before anything else
# ============================================================
import os

# Clone the repo
!git clone https://github.com/abojoudah/cs496-codeswitching.git

# Move into the repo directory
os.chdir("cs496-codeswitching")

# Verify
!ls experiments/

Cloning into 'cs496-codeswitching'...
remote: Enumerating objects: 293, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 293 (delta 64), reused 13 (delta 9), pack-reused 194 (from 1)
Receiving objects: 100% (293/293), 471.95 KiB | 5.19 MiB/s, done.
Resolving deltas: 100% (125/125), done.
data  hypotheses.md  prompts  results  scripts


## CS496 Code-Switching — Experiment Runner
Run all cells top to bottom. Only change values in the **Config** cell.

In [ ]:
# ============================================================
#  CONFIG — only edit this cell
# ============================================================

MODEL_NAME = "claude-sonnet-4-6"
API_KEY    = ""   # paste your new key here
BASE_URL   = ""             # leave empty for Anthropic

# Experiment type: "lid"
EXPERIMENT = "lid"

# Prompting strategy: "zero_shot" or "few_shot"
STRATEGY = "zero_shot"

# Sentences per API call (batch size)
BATCH_SIZE = 5

# Paths (do not change)
SAMPLE_PATH   = "experiments/data/sample_500.json"
PROMPT_PATH   = f"experiments/prompts/{STRATEGY}_{EXPERIMENT}.txt"
RESULTS_DIR   = "experiments/results"
MODEL_SLUG    = MODEL_NAME.replace("/", "-").replace(":", "-")
RAW_PATH      = f"{RESULTS_DIR}/{MODEL_SLUG}_{STRATEGY}_{EXPERIMENT}_raw.json"
PARSED_PATH   = f"{RESULTS_DIR}/{MODEL_SLUG}_{STRATEGY}_{EXPERIMENT}_parsed.json"

print(f"Model     : {MODEL_NAME}")
print(f"Provider  : {BASE_URL or 'native SDK'}")
print(f"Experiment: {EXPERIMENT}")
print(f"Strategy  : {STRATEGY}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Raw out   : {RAW_PATH}")
print(f"Parsed out: {PARSED_PATH}")

Model     : claude-sonnet-4-6
Provider  : native SDK
Experiment: lid
Strategy  : zero_shot
Batch size: 2
Raw out   : experiments/results/claude-sonnet-4-6_zero_shot_lid_raw.json
Parsed out: experiments/results/claude-sonnet-4-6_zero_shot_lid_parsed.json


## 1. Install & Imports

In [ ]:
!pip install openai anthropic google-generativeai groq -q

import os, json, time, re
from datetime import datetime


## 2. Load Data & Prompt

In [ ]:
# Load 500-sentence sample
with open(SAMPLE_PATH, "r", encoding="utf-8") as f:
    sample_data = json.load(f)

records = sample_data["records"]
print(f"Loaded {len(records)} records")
print(f"Dataset breakdown: {sample_data['dataset_breakdown']}")

# Load prompt template
with open(PROMPT_PATH, "r", encoding="utf-8") as f:
    PROMPT_TEMPLATE = f.read()

print(f"\nPrompt template loaded ({len(PROMPT_TEMPLATE)} chars)")
print("\nFirst 200 chars of prompt:")
print(PROMPT_TEMPLATE[:200])

# Create results directory
os.makedirs(RESULTS_DIR, exist_ok=True)


In [ ]:
# Remove .gitkeep so git tracks real result files
gitkeep = "experiments/results/.gitkeep"
if os.path.exists(gitkeep):
    os.remove(gitkeep)

## 3. API Caller (Model-Agnostic)

In [ ]:
def detect_provider(model_name: str, base_url: str) -> str:
    """Detect which API provider to use based on model name and base_url."""
    m = model_name.lower()
    if "claude" in m:
        return "anthropic"
    elif "gemini" in m:
        return "google"
    elif base_url:
        return "openai_compatible"  # covers GitHub Models, Groq, and any OpenAI-compatible API
    else:
        raise ValueError(f"Cannot detect provider for model: {model_name}")

PROVIDER = detect_provider(MODEL_NAME, BASE_URL)
print(f"Detected provider: {PROVIDER}")

# Initialize client
if PROVIDER == "openai_compatible":
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

elif PROVIDER == "anthropic":
    import anthropic
    client = anthropic.Anthropic(api_key=API_KEY)

elif PROVIDER == "google":
    import google.generativeai as genai
    genai.configure(api_key=API_KEY)
    client = genai.GenerativeModel(MODEL_NAME)

print(f"Client initialized for {PROVIDER}")

In [ ]:
import re

# ── Tune these if needed ─────────────────────────────────────────
QUICK_WAITS = [1, 5, 10]   # waits for attempts 1, 2, 3

def parse_retry_seconds(error_msg: str) -> float | None:
    """
    Extract the suggested wait time from a rate-limit error message.
    Handles four common formats:
      - "try again in 6m34.848s"  → Groq TPD/TPM errors
      - "try again in 45s"        → seconds-only
      - "try again in 5ms"        → milliseconds (Groq short waits)
      - "wait 85730 seconds"      → GitHub Models daily limit
    Returns None if no pattern matches.
    """
    msg = str(error_msg).lower()

    # Format: "try again in 6m34.848s"
    m = re.search(r"try again in (\d+)m([\d.]+)s", msg)
    if m:
        return int(m.group(1)) * 60 + float(m.group(2))

    # Format: "try again in 45s"
    m = re.search(r"try again in ([\d.]+)s", msg)
    if m:
        return float(m.group(1))

    # Format: "try again in 5ms"
    m = re.search(r"try again in ([\d.]+)ms", msg)
    if m:
        return float(m.group(1)) / 1000

    # Format: "wait 85730 seconds"
    m = re.search(r"wait (\d+) seconds", msg)
    if m:
        return float(m.group(1))

    return None


def call_api(prompt: str, max_retries: int = 5) -> str:
    """
    Call the API with smart retry logic.
    - Attempts 1-3: quick fixed waits (1s, 5s, 10s) for transient errors
    - Attempts 4-5: reads actual wait time from the error message
    - Fallback: exponential backoff if no wait time found in message
    """
    for attempt in range(max_retries):
        try:
            if PROVIDER == "openai_compatible":
                response = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0,
                    max_tokens=2048,
                )
                return response.choices[0].message.content

            elif PROVIDER == "anthropic":
                response = client.messages.create(
                    model=MODEL_NAME,
                    max_tokens=2048,
                    messages=[{"role": "user", "content": prompt}],
                )
                return response.content[0].text

            elif PROVIDER == "google":
                response = client.generate_content(
                    prompt,
                    generation_config={"temperature": 0.0, "max_output_tokens": 2048}
                )
                return response.text

        except Exception as e:
            error_msg = str(e)

            if attempt < len(QUICK_WAITS):
                # Early attempts: quick fixed waits
                wait = QUICK_WAITS[attempt]
                print(f"  [Attempt {attempt+1}/{max_retries}] Error: {error_msg[:80]}. Waiting {wait}s...")
            else:
                # Later attempts: read wait from error message
                suggested = parse_retry_seconds(error_msg)
                if suggested:
                    wait = suggested + 2  # small buffer
                    print(f"  [Attempt {attempt+1}/{max_retries}] Rate limit. Waiting {wait:.1f}s (from error message)...")
                else:
                    wait = 2 ** attempt
                    print(f"  [Attempt {attempt+1}/{max_retries}] Error: {error_msg[:80]}. Backoff {wait}s...")

            time.sleep(wait)

    raise RuntimeError(f"API call failed after {max_retries} attempts.")

print("call_api() defined.")

## 4. Prompt Builder

In [ ]:
def build_prompt(batch: list) -> str:
    """
    Build the prompt for a batch of records.
    Replaces the placeholder at the end of the template with actual sentences.
    """
    sentences_block = "\n".join(
        f'sent_{i+1}: "{r["text"]}"'
        for i, r in enumerate(batch)
    )
    # Replace the placeholder line at the bottom of the template
    prompt = PROMPT_TEMPLATE.replace(
        'sent_{sent_num}: "{sentence}"',
        sentences_block
    )
    return prompt

# Test it
test_prompt = build_prompt(records[:2])
print("Sample prompt (last 300 chars):")
print(test_prompt[-300:])


## 5. Response Parser

In [ ]:
VALID_LABELS = {"AR", "EN", "AR-LAT", "OTHER", "OL"}

def parse_response(raw: str, batch: list, experiment: str) -> dict:
    """
    Parse raw API response into structured output.
    Returns a dict with parsed results and a parse_status field.
    """
    result = {
        "parse_status": "ok",
        "parse_error": None,
        "sentences": []
    }

    # Strip markdown code fences if present
    cleaned = raw.strip()
    cleaned = re.sub(r"^```json\s*", "", cleaned)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"```\s*$", "", cleaned)
    cleaned = cleaned.strip()

    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError as e:
        result["parse_status"] = "json_error"
        result["parse_error"] = str(e)
        return result

    if not isinstance(parsed, list):
        result["parse_status"] = "format_error"
        result["parse_error"] = "Response is not a JSON array"
        return result

    for i, item in enumerate(parsed):
        if i >= len(batch):
            break
        record = batch[i]

        if experiment == "lid":
            tokens_out = item.get("tokens", [])

            # Check token count match
            if len(tokens_out) != record["n_tokens"]:
                result["parse_status"] = "token_count_mismatch"
                result["parse_error"] = (
                    f"sent_{i+1}: expected {record['n_tokens']} tokens, "
                    f"got {len(tokens_out)}"
                )

            # Check for invalid labels
            for t in tokens_out:
                if t.get("label") not in VALID_LABELS:
                    result["parse_status"] = "invalid_label"
                    result["parse_error"] = f"Invalid label: {t.get('label')}"

            result["sentences"].append({
                "id": record["id"],
                "dataset": record["dataset"],
                "text": record["text"],
                "predicted_tokens": tokens_out,
                "gold_labels": record["gold_labels"],
                "n_tokens_expected": record["n_tokens"],
                "n_tokens_predicted": len(tokens_out),
            })

        elif experiment == "mt":
            translation = item.get("translation", "")
            result["sentences"].append({
                "id": record["id"],
                "dataset": record["dataset"],
                "text": record["text"],
                "translation": translation,
            })

    return result

print("parse_response() defined.")


## 6. Test Run (5 Sentences)
Run this first before the full run. Check the output carefully.

In [ ]:
TEST_RECORDS = records[:BATCH_SIZE]
test_batch_prompt = build_prompt(TEST_RECORDS)

print("Calling API with 5 test sentences...")
test_raw = call_api(test_batch_prompt)

print("\nRaw response:")
print(test_raw)

print("\nParsed output:")
test_parsed = parse_response(test_raw, TEST_RECORDS, EXPERIMENT)
print(f"Parse status: {test_parsed['parse_status']}")
if test_parsed["parse_error"]:
    print(f"Parse error: {test_parsed['parse_error']}")
for s in test_parsed["sentences"]:
    print(f"\n{s['id']}: {s['text'][:60]}...")
    if EXPERIMENT == "lid":
        print(f"  Predicted: {[t['label'] for t in s['predicted_tokens']]}")
        print(f"  Gold     : {s['gold_labels']}")
    elif EXPERIMENT == "mt":
        print(f"  Translation: {s['translation']}")


## 7. Full Run (500 Sentences)
Saves each result immediately to a JSONL checkpoint. If the session crashes, re-run this cell — it will skip already-completed sentences automatically.

In [ ]:
# ── Checkpoint paths ────────────────────────────────────────────
CHECKPOINT_PATH = f"{RESULTS_DIR}/{MODEL_SLUG}_{STRATEGY}_{EXPERIMENT}_checkpoint.jsonl"

# ── Load already-completed IDs from checkpoint (resume support) ──
completed_ids = set()
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as ck:
        for line in ck:
            try:
                rec = json.loads(line)
                completed_ids.add(rec["id"])
            except:
                pass
    print(f"Resuming: {len(completed_ids)} sentences already done, skipping them.")
else:
    print("No checkpoint found. Starting fresh.")

# ── Filter out completed records ─────────────────────────────────
remaining = [r for r in records if r["id"] not in completed_ids]
print(f"Sentences to process: {len(remaining)} / {len(records)}")

# ── Split into batches ───────────────────────────────────────────
batches = [remaining[i:i+BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]
print(f"Batches to run: {len(batches)} (batch size: {BATCH_SIZE})\n")

parse_failures = 0
run_start = datetime.now().isoformat()

# ── Open checkpoint file for appending ───────────────────────────
ck_file = open(CHECKPOINT_PATH, "a", encoding="utf-8")

try:
    for batch_idx, batch in enumerate(batches):
        print(f"Batch {batch_idx+1}/{len(batches)} ({[r['id'] for r in batch]})...", end=" ")

        prompt = build_prompt(batch)

        # ── First attempt: whole batch ────────────────────────────
        try:
            raw_text = call_api(prompt)
            parsed = parse_response(raw_text, batch, EXPERIMENT)
        except RuntimeError as e:
            print(f"BATCH FAILED: {e}")
            parsed = {"parse_status": "api_error", "parse_error": str(e), "sentences": []}

        # ── If batch parse fails, retry each sentence alone ───────
        if parsed["parse_status"] != "ok" and len(batch) > 1:
            print(f"\n  Parse issue ({parsed['parse_status']}). Retrying each sentence individually...")
            parsed = {"parse_status": "ok", "parse_error": None, "sentences": []}

            for single in batch:
                print(f"    Retrying {single['id']}...", end=" ")
                try:
                    single_raw = call_api(build_prompt([single]))
                    single_parsed = parse_response(single_raw, [single], EXPERIMENT)
                    if single_parsed["parse_status"] != "ok":
                        print(f"STILL FAILED ({single_parsed['parse_status']})")
                        parse_failures += 1
                        parsed["sentences"].append({
                            "id": single["id"],
                            "dataset": single["dataset"],
                            "text": single["text"],
                            "parse_failure": True,
                            "parse_error": single_parsed["parse_error"],
                        })
                    else:
                        print("ok")
                        parsed["sentences"].extend(single_parsed["sentences"])
                    time.sleep(1)
                except RuntimeError as e:
                    print(f"API ERROR: {e}")
                    parse_failures += 1
        elif parsed["parse_status"] != "ok":
            # Single sentence and it still failed
            print(f"FAILED ({parsed['parse_status']}): {parsed['parse_error']}")
            parse_failures += len(batch)
        else:
            print("ok")

        # ── Save each sentence to JSONL immediately ───────────────
        for sentence in parsed["sentences"]:
            ck_file.write(json.dumps(sentence, ensure_ascii=False) + "\n")
        ck_file.flush()  # force write to disk now

        time.sleep(0.5)

finally:
    ck_file.close()

total_done = len(completed_ids) + len(remaining) - parse_failures
print(f"\nDone. Parse failures this run: {parse_failures}/{len(remaining)}")
print(f"Checkpoint saved to: {CHECKPOINT_PATH}")




## 8. Build Final Output Files
Reads the JSONL checkpoint and builds the standard `_parsed.json` file expected by `evaluate.py`.

In [ ]:
# ── Build final JSON output files from the JSONL checkpoint ─────
# This consolidates all sentences (including from previous runs) into
# the standard raw/parsed JSON format expected by evaluate.py

all_sentences = []
parse_failure_count = 0

with open(CHECKPOINT_PATH, "r", encoding="utf-8") as ck:
    for line in ck:
        try:
            rec = json.loads(line)
            all_sentences.append(rec)
            if rec.get("parse_failure"):
                parse_failure_count += 1
        except:
            pass

print(f"Total sentences in checkpoint: {len(all_sentences)}")
print(f"Parse failures: {parse_failure_count}")

# Save parsed output (standard format for evaluate.py)
parsed_output = {
    "model": MODEL_NAME,
    "provider": PROVIDER,
    "experiment": EXPERIMENT,
    "strategy": STRATEGY,
    "total_sentences": len(all_sentences),
    "parse_failures": parse_failure_count,
    "parse_failure_rate": round(parse_failure_count / max(len(all_sentences), 1) * 100, 2),
    "sentences": all_sentences
}

with open(PARSED_PATH, "w", encoding="utf-8") as f:
    json.dump(parsed_output, f, ensure_ascii=False, indent=2)

print(f"\nParsed output saved to: {PARSED_PATH}")
print(f"Parse failure rate: {parsed_output['parse_failure_rate']}%")
print("Ready to run evaluate.py")


## 9. Push Results to GitHub

In [ ]:
GITHUB_TOKEN = "your_token_here"
GITHUB_USER  = "abojoudah"
GITHUB_REPO  = "cs496-codeswitching"

!git add experiments/results/
!git commit -m "Add {MODEL_SLUG}_{STRATEGY}_{EXPERIMENT} results"
!git push https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git main
